# N-Grams

Experimenting with different methodologies on utilizing both the infinigram API and GPT2, as well as Pythia/OLMO checkpoints.

In [ ]:
# ### QUICK TEST
#
# from transformer_lens import HookedTransformer
#
# # load a tiny model (70M params)
# model = HookedTransformer.from_pretrained("EleutherAI/pythia-70m")
# tok = model.tokenizer
#
# # test it
# tokens = tok("Hello world, this is a test", return_tensors="pt")
# out = model.run_with_cache(tokens.input_ids)
# # out["logits"] etc. are now available for your n-gram / activation dumps

In [ ]:
# ! aws s3 cp --no-sign-request --recursive s3://infini-gram-lite/index/v4_pileval_llama data/corpus/v4_pileval_gpt2

In [ ]:
from transformer_lens import HookedTransformer
from infini_gram.engine import InfiniGramEngine
import torch
import collections

In [ ]:
model = HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer

In [ ]:
engine = InfiniGramEngine(
  index_dir="./data/corpus/v4_pileval_gpt2",
  eos_token_id=tokenizer.eos_token_id
)

In [ ]:
# Replace this with any longer text (list of strings) or a DataLoader
text = "In the field of machine learning, transformers have revolutionized NLP."
ids = tokenizer(text, return_tensors="pt", padding=False).input_ids[0]  # [batch, seq_len]

logits, cache = model.run_with_cache(ids.unsqueeze(0))

In [ ]:
layer = 4
acts  = cache["resid_post", layer][0]

In [ ]:
# find the token ID for " the"
tid = tokenizer.encode(" the", add_special_tokens=False)
print("token IDs:", tid)              # e.g. [464]

# count it
res = engine.count(input_ids=tid)
print(res)

In [ ]:
import torch

# pick a feature i
i = 0
# find its top‑5 spikes
vals, idxs = torch.topk(acts[:, i], k=5)   # vals unused
for pos in idxs.tolist():
    token_id = int(ids[pos])
    cnt      = engine.count(input_ids=[token_id])["count"]
    print(f"neuron {i} spike at pos {pos} on token", tokenizer.decode([token_id]), "→ count", cnt)


In [ ]:
k = 5
d_model = acts.shape[1]

# 1a) find top‑k spike positions for all features
vals, idxs = torch.topk(acts, k, dim=0)   # idxs: [k, d_model]

# 1b) for each feature, build its windows and count
n = 1
feature_counts = {}
for feat in range(d_model):
    cnts = []
    for pos in idxs[:, feat].tolist():
        start = max(0, pos - (n-1)//2)
        end   = min(len(ids), pos + (n//2) + 1)
        window_ids = ids[start:end].tolist()      # now 3‑gram
        cnts.append(engine.count(input_ids=window_ids)["count"])
    feature_counts[feat] = cnts

In [ ]:
print(feature_counts)

In [ ]:
import numpy as np

feature_summary = {}
for feat, cnts in feature_counts.items():
    feature_summary[feat] = {
        "max_count":   max(cnts),
        "mean_count":  np.mean(cnts),
        "med_count":   np.median(cnts),
    }

In [ ]:
for feat, summary in feature_summary.items():
    summary["log1p_max"]  = np.log1p(summary["max_count"])
    summary["log1p_mean"] = np.log1p(summary["mean_count"])

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(feature_summary, orient="index")
df.index.name = "feature"
df.reset_index(inplace=True)
df.to_csv("layer4_1gram_freq_summary.csv", index=False)

In [ ]:
print(df)